# Audio QnA APP - RAG
It does not require GPU access to run this notebook.

This notebook is authored by [Anmol Talwar](https://www.linkedin.com/in/anmol-talwar-922061164/) - Founder and Trainer at Talent Catalyst AI

Visit the blogs on [Talent Catalyst AI](https://talentcatalystai.com/) to learn more on New Gen Technology.

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers.

YT Video Link :  https://www.youtube.com/watch?v=rfr01atilTM

### Installing required python packages

In [ ]:
!pip -q install sentence_transformers faiss-cpu langchain text-generation openai azure.identity InstructorEmbedding langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

### Importing librarires

In [ ]:
import os
import openai
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.llms import HuggingFaceHub
from text_generation import Client, InferenceAPIClient
from functools import partial
from langchain_community.embeddings import HuggingFaceInstructEmbeddings
from openai import OpenAI

/tmp/ipykernel_2430/1564645489.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


### Setting API tokens

In [ ]:
api_key = "YOUR_ACTUAL_OPENAI_KEY"
client = OpenAI(api_key=api_key)

os.environ["HUGGING_FACE_HUB_TOKEN"] = "YOUR_ACTUAL_HF_TOKEN"

In [ ]:
import os
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

os.environ["HUGGING_FACE_HUB_TOKEN"] = os.getenv("HUGGING_FACE_HUB_TOKEN")

**Speech - to - Text**

In [ ]:
def STT(path):
  audio_file= open(path, "rb")
  transcription = client.audio.transcriptions.create(
  model="whisper-1",
  file=audio_file)

  file_path = "/content/temp.txt"

  with open(file_path, "w") as text_file:
      # Write the text to the file
      text_file.write(transcription.text)

  print("Text has been saved to:", file_path)

  return file_path

### Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size = 1000,
    chunk_overlap  = 50,
    length_function = len,
    separators=["\n\n", "\n", " ", ""]
)

### Data Embedding

In [ ]:
hf_embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
hf_embeddings = HuggingFaceEmbeddings(model_name=hf_embedding_model_name)

/tmp/ipykernel_2430/840584322.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(model_name=hf_embedding_model_name)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### LLM Request

In [ ]:
def call_gpt_35_usllm(prompt):

  response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"{prompt}"}]
  )

  return response.choices[0].message.content

### Integrating All Components

In [ ]:
def run_app(db, query):


  contexts = db.similarity_search(query, k=5)

  context = ''
  for n in range(len(contexts)):
      context += contexts[n].page_content
  #         print(context)


  prompt = f'''Act as an analyst and answer the question as perfectly as possible (in a consice manner) from the conversation between a call center agent and customer. Strictly donot answer any questions except for the context\n\n\
                Question: {query}\n\
                Context: {context}\n\
                Response: '''
  print(prompt)

  response = call_gpt_35_usllm(prompt)
  # print(response)
  # n = '_'.join(str(llm.func).split('_')[1:-1])
  # print(f'\n>>> Response from Embedding model:all-MiniLM-L6-v2 & LLM:{n}\n',response.strip())

  return response.strip()

### Data Storage (Vector Database)

In [ ]:
def embedding_store(file, embedding):

    with open(file, "r") as text_file:
      # Read the contents of the file
      text = text_file.read()

    texts =  text_splitter.split_text(text)
    # converting text chunks into Document chunks
    docs = []
    for i, chunk in enumerate(texts):
        doc = Document(
            page_content = chunk,
            metadata = {
                "page": i + 1,
                "chunk": i}
        )
        # Add sources a metadata
        doc.metadata["source"] = f"{doc.metadata['page']}-{doc.metadata['chunk']}"
        docs.append(doc)
    #st.write(texts)
    docsearch = FAISS.from_documents(docs, embedding)
    print("Vector DB Processed")

    return docsearch

In [ ]:
audio_path = '/content/campaign-call-center.mp3.mpeg'


In [ ]:
# USE ONLY FIRST TIME
file = STT(audio_path)
docsearch = embedding_store(file, hf_embeddings)

Text has been saved to: /content/temp.txt
Vector DB Processed


In [ ]:
query = "What is the name of the customer?"

response_final = run_app(
        db=docsearch, query = query
    )
response_final

Act as an analyst and answer the question as perfectly as possible (in a consice manner) from the conversation between a call center agent and customer. Strictly donot answer any questions except for the context

                Question: What is the name of the customer?
                Context: answer any questions, find the plan that best works for you. Is it okay if I connect you? Yeah. All right. So you may hear some ringing in the background. As soon as they pick up, I'll introduce you, and then you'll be on your way, okay? Okay. Hello, this is Al in the final expense department. Who do I have the pleasure of speaking with? Hello, Al. Hi, Irene.or statement account. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Okay. Thank you all for speaking on the line. There is one question left for you. I understand that you're looking for the final expense codes, okay? Just a second. Let me just bring them on the line. Here we go

'The name of the customer is Irene Proctor.'

In [ ]:
query = "Age of the customer?"

response_final = run_app(
        db=docsearch, query = query
    )
response_final

Act as an analyst and answer the question as perfectly as possible (in a consice manner) from the conversation between a call center agent and customer. Strictly donot answer any questions except for the context

                Question: Age of the customer?
                Context: answer any questions, find the plan that best works for you. Is it okay if I connect you? Yeah. All right. So you may hear some ringing in the background. As soon as they pick up, I'll introduce you, and then you'll be on your way, okay? Okay. Hello, this is Al in the final expense department. Who do I have the pleasure of speaking with? Hello, Al. Hi, Irene.Hello. Hello. Yes. Hi. My name is Brian Anderson calling you back from Senior Benefits. How are you doing today? Great. How are you? I'm good. I'm good. Thank you so much for asking. Well, we have words with you yesterday, and you asked me to call you back, and the best callback time is morning, and your favorite color is blue, right? Yes. Okay. And yo

'The age of the customer is 79 years old, born in 1943.'

**Gradio APP**

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import time

with gr.Blocks(css=".gradio-container {background-color: Grey}") as demo:

    gr.Markdown("## Education Made Easy :  Custom Video QnA Chat-Bot")
    gr.Image("/content/TCAI_LOGO_F.png", label="Talent Catalyst AI", width=300, height=200)

    chatbot = gr.Chatbot(label = "Audio QnA App")
    msg = gr.Textbox(label = "Ask your Question here")
    clear = gr.ClearButton([msg, chatbot])

    def respond(message, chat_history):
        bot_message = run_app(db = docsearch, query = message)
        chat_history.append((message, bot_message))
        time.sleep(2)
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_2430/1282280412.py:4: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=".gradio-container {background-color: Grey}") as demo:
/tmp/ipykernel_2430/1282280412.py:9: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label = "Audio QnA App")
/tmp/ipykernel_2430/1282280412.py:9: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label = "Audio QnA App")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a845ee737827a61c89.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')